# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoriful-mynul/flyrank-assignment1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)
print("Connection OK")

DuckDB version: 1.3.2
Connection OK


## 1. Question

Can historical search-performance signals help prioritize content pages that are most worth reviewing or refreshing?

## Decision framing

Which existing content pages should a content team review first, based only on information available before the review decision?

### Project objective

The objective is to identify content pages that are most likely to experience
meaningful search-performance decline and rank them for content-refresh
prioritization using information available at decision time.

## 2. Data

### Dataset and source

This capstone uses the pseudonymized FlyRank internship warehouse release
`flyrank_pseudonymized_warehouse_release_v20260703`, exported on 2026-07-03.

The analysis uses the following warehouse tables:

- `dim_clients` for client-level metadata.
- `dim_content` for content-page metadata.
- `fact_content_daily_performance` for daily search and traffic performance signals.

The warehouse contains daily content-performance data from 2025-01-27
through 2026-06-30. The final capstone analysis uses March 2026 as the
decision window and April 2026 as the future evaluation window.

### Decision-time population

The decision date is 2026-03-31. Only content that existed by this date
was retained. Pages younger than 90 days were excluded so that very new
content would not be treated as established content requiring refresh
prioritization.

For supervised target construction, pages needed at least 500 March
Google Search Console impressions and observable Google Search Console
coverage in both the decision and future periods.

The final deployment-style ranking does not use April information. It
scores March-visible mature pages with at least 500 March impressions.

### Public-safe handling

Client and content identifiers remain pseudonymized. No client names,
domains, private search queries, credentials, or raw warehouse exports are
included in the analysis output.

In [3]:
# Check available schemas and tables

tables = con.sql("""
    SHOW ALL TABLES
""").df()

tables

,database,schema,name,column_names,column_types,temporary


In [4]:
# Check whether Hugging Face access is available

try:
    import huggingface_hub
    print("huggingface_hub:", huggingface_hub.__version__)
except Exception as e:
    print("Hugging Face library error:", e)

huggingface_hub: 1.28.0


In [5]:
from huggingface_hub import HfApi

api = HfApi()

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset found:", info.id)
    print("Access OK")
except Exception as e:
    print("Access error:", type(e).__name__)
    print(e)

Dataset found: FlyRank/internship-warehouse
Access OK


In [6]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files[:100]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [8]:
import os
from huggingface_hub import get_token

hf_token = get_token()

print("HF token available:", hf_token is not None)

HF token available: True


In [9]:
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


In [11]:
print("Clients columns:")
print(clients.columns)

print("\nContent columns:")
print(content.columns)

Clients columns:
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']

Content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [13]:
# Step 1: Check whether DuckDB can access the March 2026 parquet file

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

try:
    march = con.read_parquet(march_path)

    print("March data loaded successfully!")
    print("Shape:", march.shape)
    print("\nColumns:")
    print(march.columns)

except Exception as e:
    print("March data load failed.")
    print("Error type:", type(e).__name__)
    print("Error:")
    print(e)

March data loaded successfully!
Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [14]:
# Step 2: Check March 2026 data coverage and basic integrity

print("Date coverage:")
print(
    con.sql("""
        SELECT
            MIN(report_date) AS min_date,
            MAX(report_date) AS max_date,
            COUNT(DISTINCT report_date) AS unique_dates
        FROM march
    """).df()
)

print("\nUnique clients:")
print(
    con.sql("""
        SELECT COUNT(DISTINCT client_hash_id) AS unique_clients
        FROM march
    """).df()
)

print("\nUnique content pages:")
print(
    con.sql("""
        SELECT COUNT(DISTINCT content_hash_id) AS unique_content_pages
        FROM march
    """).df()
)

Date coverage:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  unique_dates
0 2026-03-01 2026-03-31            31

Unique clients:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   unique_clients
0              55

Unique content pages:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   unique_content_pages
0                331437


In [15]:
# Step 3: Check the grain of the March daily performance data

print("Rows:", march.shape[0])

print("\nDistinct client-page-date combinations:")
print(
    con.sql("""
        SELECT COUNT(*) AS distinct_combinations
        FROM (
            SELECT DISTINCT
                client_hash_id,
                content_hash_id,
                report_date
            FROM march
        )
    """).df()
)

print("\nDuplicate client-page-date rows:")
print(
    con.sql("""
        SELECT
            COUNT(*) - COUNT(DISTINCT
                client_hash_id || '|' ||
                content_hash_id || '|' ||
                CAST(report_date AS VARCHAR)
            ) AS duplicate_rows
        FROM march
    """).df()
)

Rows: 9841378

Distinct client-page-date combinations:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   distinct_combinations
0                9841378

Duplicate client-page-date rows:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   duplicate_rows
0               0


In [16]:
# Step 4: Check GSC and GA4 data availability

availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
            AS gsc_available_rows,

        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            AS ga4_available_rows,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                 AND ga4_data_available IS TRUE
                THEN 1 ELSE 0
            END
        ) AS both_available_rows

    FROM march
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061.0,413966.0,364347.0


In [17]:
# Step 5: Check coverage of key performance signals

coverage = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)
            AS rows_with_impressions,

        SUM(CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END)
            AS rows_with_clicks,

        SUM(CASE WHEN ga4_sessions > 0 THEN 1 ELSE 0 END)
            AS rows_with_sessions,

        SUM(CASE WHEN ga4_pageviews > 0 THEN 1 ELSE 0 END)
            AS rows_with_pageviews,

        SUM(CASE WHEN scroll_events > 0 THEN 1 ELSE 0 END)
            AS rows_with_scroll_events

    FROM march
""").df()

coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_impressions,rows_with_clicks,rows_with_sessions,rows_with_pageviews,rows_with_scroll_events
0,9841378,3611061.0,417981.0,410335.0,413317.0,123040.0


In [18]:
# Step 6: Verify that March performance rows match content metadata

join_check = con.sql("""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(c.content_hash_id) AS matched_content_rows,
        COUNT(*) - COUNT(c.content_hash_id) AS unmatched_content_rows
    FROM march m
    LEFT JOIN content c
        ON m.client_hash_id = c.client_hash_id
       AND m.content_hash_id = c.content_hash_id
""").df()

join_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,matched_content_rows,unmatched_content_rows
0,9841378,9841378,0


In [19]:
# Step 7: Check content status and metadata coverage

content_status = con.sql("""
    SELECT
        COUNT(*) AS total_content_rows,

        SUM(CASE WHEN is_published IS TRUE THEN 1 ELSE 0 END)
            AS published_rows,

        SUM(CASE WHEN is_deleted IS TRUE THEN 1 ELSE 0 END)
            AS deleted_rows,

        SUM(CASE WHEN content_created_date IS NOT NULL THEN 1 ELSE 0 END)
            AS has_created_date,

        SUM(CASE WHEN last_optimized_date IS NOT NULL THEN 1 ELSE 0 END)
            AS has_last_optimized_date

    FROM content
""").df()

content_status

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_content_rows,published_rows,deleted_rows,has_created_date,has_last_optimized_date
0,519606,411540.0,101559.0,519606.0,45396.0


In [21]:
# Step 8: Check page-level GSC performance coverage in March

march_page_coverage = con.sql("""
    SELECT
        COUNT(DISTINCT content_hash_id) AS total_pages,

        COUNT(DISTINCT CASE
            WHEN gsc_impressions > 0
            THEN content_hash_id
        END) AS pages_with_impressions,

        COUNT(DISTINCT CASE
            WHEN gsc_clicks > 0
            THEN content_hash_id
        END) AS pages_with_clicks

    FROM march
""").df()

march_page_coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_pages,pages_with_impressions,pages_with_clicks
0,331437,176738,68837


In [22]:
# Step 9: Check daily observation coverage per content page

daily_coverage = con.sql("""
    SELECT
        COUNT(*) AS page_count,
        MIN(days_observed) AS min_days_observed,
        MAX(days_observed) AS max_days_observed,
        AVG(days_observed) AS avg_days_observed
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS days_observed
        FROM march
        GROUP BY
            client_hash_id,
            content_hash_id
    )
""").df()

daily_coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,page_count,min_days_observed,max_days_observed,avg_days_observed
0,331437,1,31,29.693058


In [23]:
# Step 10: Check whether April 2026 data is available for future-target construction

april_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

try:
    april = con.read_parquet(april_path)

    print("April data loaded successfully!")
    print("Shape:", april.shape)

    print("\nDate coverage:")
    print(
        con.sql("""
            SELECT
                MIN(report_date) AS min_date,
                MAX(report_date) AS max_date,
                COUNT(DISTINCT report_date) AS unique_dates
            FROM april
        """).df()
    )

except Exception as e:
    print("April data load failed.")
    print("Error type:", type(e).__name__)
    print("Error:")
    print(e)

April data loaded successfully!
Shape: (10424730, 30)

Date coverage:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  unique_dates
0 2026-04-01 2026-04-30            30


In [24]:
# Step 11: Check March-to-April page continuity

continuity_check = con.sql("""
    SELECT
        COUNT(DISTINCT m.content_hash_id) AS march_pages,

        COUNT(DISTINCT CASE
            WHEN a.content_hash_id IS NOT NULL
            THEN m.content_hash_id
        END) AS pages_present_in_april,

        COUNT(DISTINCT CASE
            WHEN a.content_hash_id IS NULL
            THEN m.content_hash_id
        END) AS pages_missing_in_april

    FROM march m
    LEFT JOIN (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM april
    ) a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
""").df()

continuity_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_pages,pages_present_in_april,pages_missing_in_april
0,331437,331436,1


In [25]:
# Step 12: Check future-period (April) GSC performance coverage

april_coverage = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE
            WHEN gsc_data_available IS TRUE
            THEN 1 ELSE 0
        END) AS gsc_available_rows,

        SUM(CASE
            WHEN gsc_impressions > 0
            THEN 1 ELSE 0
        END) AS rows_with_impressions,

        SUM(CASE
            WHEN gsc_clicks > 0
            THEN 1 ELSE 0
        END) AS rows_with_clicks

    FROM april
""").df()

april_coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,rows_with_impressions,rows_with_clicks
0,10424730,3901060.0,3901060.0,408047.0


In [26]:
# Step 13: Examine March-to-April impression changes

impression_change = con.sql("""
    WITH march_pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM march
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april_pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM april
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        COUNT(*) AS comparable_pages,

        AVG(march_impressions) AS avg_march_impressions,
        AVG(april_impressions) AS avg_april_impressions,

        MEDIAN(march_impressions) AS median_march_impressions,
        MEDIAN(april_impressions) AS median_april_impressions,

        AVG(
            CASE
                WHEN march_impressions > 0
                THEN (april_impressions - march_impressions)
                     / march_impressions
            END
        ) AS avg_relative_change

    FROM march_pages m
    INNER JOIN april_pages a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    WHERE march_impressions > 0
""").df()

impression_change

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,comparable_pages,avg_march_impressions,avg_april_impressions,median_march_impressions,median_april_impressions,avg_relative_change
0,158549,1768.034671,1777.749194,246.0,200.0,1.954214


In [27]:
# Step 14: Compare candidate decline thresholds

decline_thresholds = con.sql("""
    WITH march_pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM march
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april_pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM april
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    comparable AS (
        SELECT
            m.client_hash_id,
            m.content_hash_id,
            m.march_impressions,
            a.april_impressions,
            (a.april_impressions - m.march_impressions)
                / m.march_impressions AS relative_change
        FROM march_pages m
        INNER JOIN april_pages a
            ON m.client_hash_id = a.client_hash_id
           AND m.content_hash_id = a.content_hash_id
        WHERE m.march_impressions >= 100
    )

    SELECT
        COUNT(*) AS eligible_pages,

        SUM(CASE WHEN relative_change <= -0.10 THEN 1 ELSE 0 END)
            AS decline_10pct,

        SUM(CASE WHEN relative_change <= -0.20 THEN 1 ELSE 0 END)
            AS decline_20pct,

        SUM(CASE WHEN relative_change <= -0.30 THEN 1 ELSE 0 END)
            AS decline_30pct,

        SUM(CASE WHEN relative_change <= -0.50 THEN 1 ELSE 0 END)
            AS decline_50pct

    FROM comparable
""").df()

decline_thresholds

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,decline_10pct,decline_20pct,decline_30pct,decline_50pct
0,100893,59588.0,51985.0,43342.0,25577.0


In [28]:
# Step 15: Check 20% decline rate at different minimum-demand levels

target_sensitivity = con.sql("""
    WITH march_pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM march
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april_pages AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM april
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    comparable AS (
        SELECT
            m.client_hash_id,
            m.content_hash_id,
            m.march_impressions,
            a.april_impressions,
            (a.april_impressions - m.march_impressions)
                / m.march_impressions AS relative_change
        FROM march_pages m
        INNER JOIN april_pages a
            ON m.client_hash_id = a.client_hash_id
           AND m.content_hash_id = a.content_hash_id
        WHERE m.march_impressions > 0
    )

    SELECT
        '100+' AS minimum_impressions,
        COUNT(*) FILTER (WHERE march_impressions >= 100) AS eligible_pages,
        COUNT(*) FILTER (
            WHERE march_impressions >= 100
              AND relative_change <= -0.20
        ) AS declining_pages

    FROM comparable

    UNION ALL

    SELECT
        '250+' AS minimum_impressions,
        COUNT(*) FILTER (WHERE march_impressions >= 250),
        COUNT(*) FILTER (
            WHERE march_impressions >= 250
              AND relative_change <= -0.20
        )
    FROM comparable

    UNION ALL

    SELECT
        '500+' AS minimum_impressions,
        COUNT(*) FILTER (WHERE march_impressions >= 500),
        COUNT(*) FILTER (
            WHERE march_impressions >= 500
              AND relative_change <= -0.20
        )
    FROM comparable

    UNION ALL

    SELECT
        '1000+' AS minimum_impressions,
        COUNT(*) FILTER (WHERE march_impressions >= 1000),
        COUNT(*) FILTER (
            WHERE march_impressions >= 1000
              AND relative_change <= -0.20
        )
    FROM comparable
""").df()

target_sensitivity["decline_rate"] = (
    target_sensitivity["declining_pages"]
    / target_sensitivity["eligible_pages"]
)

target_sensitivity

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,minimum_impressions,eligible_pages,declining_pages,decline_rate
0,100+,100893,51985,0.515249
1,250+,78992,40252,0.509571
2,500+,61846,30928,0.500081
3,1000+,45026,21900,0.486386


In [29]:
# Step 16A: Confirm the decision cutoff and future target window

print("Decision window:")
print(
    con.sql("""
        SELECT
            MIN(report_date) AS decision_start,
            MAX(report_date) AS decision_end
        FROM march
    """).df()
)

print("\nFuture target window:")
print(
    con.sql("""
        SELECT
            MIN(report_date) AS target_start,
            MAX(report_date) AS target_end
        FROM april
    """).df()
)

Decision window:
  decision_start decision_end
0     2026-03-01   2026-03-31

Future target window:
  target_start target_end
0   2026-04-01 2026-04-30


In [30]:
# Step 17: Build the March page-level feature frame

march_features = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Search performance
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        AVG(gsc_avg_position) AS avg_position_30d,

        -- Analytics performance
        SUM(ga4_pageviews) AS pageviews_30d,
        SUM(ga4_sessions) AS sessions_30d,
        SUM(ga4_users) AS users_30d,
        SUM(ga4_engaged_sessions) AS engaged_sessions_30d,

        -- AI / traffic signals
        SUM(sessions_ai) AS ai_sessions_30d,
        SUM(scroll_events) AS scroll_events_30d,

        -- Observation / availability
        COUNT(DISTINCT report_date) AS days_observed,

        MAX(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN 1 ELSE 0
            END
        ) AS has_gsc_data,

        MAX(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN 1 ELSE 0
            END
        ) AS has_ga4_data

    FROM march

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", march_features.shape)
print("\nColumns:")
print(march_features.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 14)

Columns:
['client_hash_id', 'content_hash_id', 'impressions_30d', 'clicks_30d', 'avg_position_30d', 'pageviews_30d', 'sessions_30d', 'users_30d', 'engaged_sessions_30d', 'ai_sessions_30d', 'scroll_events_30d', 'days_observed', 'has_gsc_data', 'has_ga4_data']


In [31]:
# Step 18: Join March performance features with content metadata

feature_frame = con.sql("""
    SELECT
        m.*,

        -- Content metadata
        c.content_created_date,
        c.content_updated_date,
        c.content_type,
        c.search_volume,
        c.competition,
        c.competition_level,
        c.cpc,
        c.main_intent,
        c.backlinks,
        c.category_count,
        c.char_count,
        c.word_count,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.is_published,
        c.is_deleted

    FROM march_features m

    INNER JOIN content c
        ON m.client_hash_id = c.client_hash_id
       AND m.content_hash_id = c.content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)

print("\nColumns:")
print(feature_frame.columns.tolist())

print("\nMissing content metadata rows:")
print(
    feature_frame[
        ["content_created_date", "content_type", "search_volume", "word_count"]
    ].isna().sum()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 30)

Columns:
['client_hash_id', 'content_hash_id', 'impressions_30d', 'clicks_30d', 'avg_position_30d', 'pageviews_30d', 'sessions_30d', 'users_30d', 'engaged_sessions_30d', 'ai_sessions_30d', 'scroll_events_30d', 'days_observed', 'has_gsc_data', 'has_ga4_data', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

Missing content metadata rows:
content_created_date         0
content_type                 0
search_volume            59407
word_count              107429
dtype: int64


In [32]:
# Step 19: Investigate missing search volume and word count

missingness_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END)
            AS missing_search_volume,

        SUM(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END)
            AS missing_word_count,

        SUM(CASE
            WHEN search_volume IS NULL
             AND word_count IS NULL
            THEN 1 ELSE 0
        END) AS missing_both,

        SUM(CASE
            WHEN is_published IS TRUE
             AND search_volume IS NULL
            THEN 1 ELSE 0
        END) AS unpublished_search_volume_missing,

        SUM(CASE
            WHEN is_deleted IS TRUE
             AND search_volume IS NULL
            THEN 1 ELSE 0
        END) AS deleted_search_volume_missing

    FROM feature_frame
""").df()

missingness_check

,total_rows,missing_search_volume,missing_word_count,missing_both,unpublished_search_volume_missing,deleted_search_volume_missing
0,331437,59407.0,107429.0,7500.0,50289.0,5287.0


In [33]:
# Step 20: Create content age and freshness features

feature_frame["content_created_date"] = pd.to_datetime(
    feature_frame["content_created_date"]
)

feature_frame["last_optimized_date"] = pd.to_datetime(
    feature_frame["last_optimized_date"]
)

decision_date = pd.Timestamp("2026-03-31")

feature_frame["content_age_days"] = (
    decision_date - feature_frame["content_created_date"]
).dt.days

feature_frame["days_since_last_optimized"] = (
    decision_date - feature_frame["last_optimized_date"]
).dt.days

print(
    feature_frame[
        [
            "content_age_days",
            "days_since_last_optimized"
        ]
    ].describe()
)

       content_age_days  days_since_last_optimized
count     331437.000000               42517.000000
mean         207.795521                 -69.938236
std          120.549978                  16.202675
min           -5.000000                 -97.000000
25%          103.000000                 -83.000000
50%          220.000000                 -72.000000
75%          278.000000                 -56.000000
max          494.000000                 -24.000000


In [34]:
# Step 21: Investigate future-dated optimization timestamps

future_optimization = con.sql("""
    SELECT
        COUNT(*) AS total_with_optimization_date,

        SUM(
            CASE
                WHEN last_optimized_date > DATE '2026-03-31'
                THEN 1 ELSE 0
            END
        ) AS future_dated_rows,

        MIN(last_optimized_date) AS earliest_optimization_date,
        MAX(last_optimized_date) AS latest_optimization_date

    FROM feature_frame
    WHERE last_optimized_date IS NOT NULL
""").df()

future_optimization

,total_with_optimization_date,future_dated_rows,earliest_optimization_date,latest_optimization_date
0,42517,42517.0,2026-04-24,2026-07-06


In [35]:
# Step 22: Check for content created after the March decision date

future_created = con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE
                WHEN content_created_date > DATE '2026-03-31'
                THEN 1 ELSE 0
            END
        ) AS future_created_rows,

        MIN(content_created_date) AS earliest_created_date,
        MAX(content_created_date) AS latest_created_date

    FROM feature_frame
""").df()

future_created

,total_rows,future_created_rows,earliest_created_date,latest_created_date
0,331437,2124.0,2024-11-22,2026-04-05


In [36]:
# Step 23: Keep only pages that existed by the March 31 decision date

eligible_features = feature_frame[
    feature_frame["content_created_date"] <= pd.Timestamp("2026-03-31")
].copy()

print("Original feature rows:", len(feature_frame))
print("Eligible rows:", len(eligible_features))
print("Excluded future-created rows:", len(feature_frame) - len(eligible_features))

print("\nContent age check:")
print(
    eligible_features["content_age_days"].describe()
)

Original feature rows: 331437
Eligible rows: 329313
Excluded future-created rows: 2124

Content age check:
count    329313.000000
mean        209.146614
std         119.754632
min           0.000000
25%         104.000000
50%         221.000000
75%         278.000000
max         494.000000
Name: content_age_days, dtype: float64


In [37]:
# Step 24: Check the age distribution of eligible content pages

age_distribution = con.sql("""
    SELECT
        CASE
            WHEN content_age_days < 30 THEN '0-29 days'
            WHEN content_age_days < 60 THEN '30-59 days'
            WHEN content_age_days < 90 THEN '60-89 days'
            WHEN content_age_days < 180 THEN '90-179 days'
            WHEN content_age_days < 365 THEN '180-364 days'
            ELSE '365+ days'
        END AS age_group,

        COUNT(*) AS page_count

    FROM eligible_features

    GROUP BY age_group

    ORDER BY
        CASE age_group
            WHEN '0-29 days' THEN 1
            WHEN '30-59 days' THEN 2
            WHEN '60-89 days' THEN 3
            WHEN '90-179 days' THEN 4
            WHEN '180-364 days' THEN 5
            WHEN '365+ days' THEN 6
        END
""").df()

age_distribution["percentage"] = (
    age_distribution["page_count"]
    / age_distribution["page_count"].sum()
    * 100
)

age_distribution

,age_group,page_count,percentage
0,0-29 days,26505,8.048574
1,30-59 days,27215,8.264174
2,60-89 days,24534,7.450055
3,90-179 days,36638,11.125586
4,180-364 days,181930,55.245314
5,365+ days,32491,9.866297


In [38]:
# Step 25: Finalize the decision-time eligible population

model_population = eligible_features[
    eligible_features["content_age_days"] >= 90
].copy()

print("Decision-time eligible pages:", len(model_population))
print(
    "Excluded pages younger than 90 days:",
    len(eligible_features) - len(model_population)
)

print("\nAge summary:")
print(
    model_population["content_age_days"].describe()
)

Decision-time eligible pages: 251059
Excluded pages younger than 90 days: 78254

Age summary:
count    251059.000000
mean        260.475008
std          86.900092
min          95.000000
25%         200.000000
50%         245.000000
75%         328.000000
max         494.000000
Name: content_age_days, dtype: float64


In [39]:
# Step 26: Feature sanity checks

df = model_population.copy()

# Derived ratios
df["ctr_30d"] = (
    df["clicks_30d"] / df["impressions_30d"].replace(0, pd.NA)
)

df["engagement_rate_30d"] = (
    df["engaged_sessions_30d"] / df["sessions_30d"].replace(0, pd.NA)
)

df["ai_traffic_share_30d"] = (
    df["ai_sessions_30d"] / df["sessions_30d"].replace(0, pd.NA)
)

df["scroll_rate_30d"] = (
    df["scroll_events_30d"] / df["pageviews_30d"].replace(0, pd.NA)
)

print("Rows checked:", len(df))

print("\nNegative metric counts:")
for col in [
    "impressions_30d",
    "clicks_30d",
    "pageviews_30d",
    "sessions_30d",
    "users_30d",
    "engaged_sessions_30d",
    "ai_sessions_30d",
    "scroll_events_30d",
]:
    print(f"{col}: {(df[col] < 0).sum()}")

print("\nRatio sanity checks:")
print("CTR > 100%:", (df["ctr_30d"] > 1).sum())
print("Engagement rate > 100%:", (df["engagement_rate_30d"] > 1).sum())
print("AI traffic share > 100%:", (df["ai_traffic_share_30d"] > 1).sum())
print("Scroll rate > 100%:", (df["scroll_rate_30d"] > 1).sum())

print("\nPosition sanity:")
print("Avg position <= 0:", (df["avg_position_30d"] <= 0).sum())

print("\nDays observed:")
print(df["days_observed"].describe())

Rows checked: 251059

Negative metric counts:
impressions_30d: 0
clicks_30d: 0
pageviews_30d: 0
sessions_30d: 0
users_30d: 0
engaged_sessions_30d: 0
ai_sessions_30d: 0
scroll_events_30d: 0

Ratio sanity checks:
CTR > 100%: 0
Engagement rate > 100%: 0
AI traffic share > 100%: 96
Scroll rate > 100%: 29

Position sanity:
Avg position <= 0: 1033

Days observed:
count    251059.000000
mean         30.769441
std           0.812977
min           2.000000
25%          31.000000
50%          31.000000
75%          31.000000
max          31.000000
Name: days_observed, dtype: float64


In [40]:
# Step 27: Investigate anomalous feature values

print("=== AI traffic share anomalies ===")
ai_anomaly = df[df["ai_traffic_share_30d"] > 1][
    [
        "client_hash_id",
        "content_hash_id",
        "sessions_30d",
        "ai_sessions_30d",
        "ai_traffic_share_30d",
    ]
]

print(ai_anomaly.head(10).to_string(index=False))
print("Count:", len(ai_anomaly))

print("\n=== Scroll rate anomalies ===")
scroll_anomaly = df[df["scroll_rate_30d"] > 1][
    [
        "client_hash_id",
        "content_hash_id",
        "pageviews_30d",
        "scroll_events_30d",
        "scroll_rate_30d",
    ]
]

print(scroll_anomaly.head(10).to_string(index=False))
print("Count:", len(scroll_anomaly))

print("\n=== Average position anomalies ===")
position_anomaly = df[df["avg_position_30d"] <= 0][
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_30d",
        "avg_position_30d",
        "has_gsc_data",
    ]
]

print(position_anomaly.head(10).to_string(index=False))
print("Count:", len(position_anomaly))

=== AI traffic share anomalies ===
         client_hash_id          content_hash_id  sessions_30d  ai_sessions_30d ai_traffic_share_30d
client_3197e6291363b4db content_ec7cab58eeae9324           1.0              2.0                  2.0
client_a60a11451483af1c content_2690ece45b4712f9           1.0              2.0                  2.0
client_a60a11451483af1c content_2d2052ebe224dec3           1.0              3.0                  3.0
client_a60a11451483af1c content_42f8001078199c5d           1.0              4.0                  4.0
client_d211cb07b9059bab content_17970b4f4f43a783           1.0              5.0                  5.0
client_d211cb07b9059bab content_23a5f572d8917ea4           2.0              4.0                  2.0
client_d211cb07b9059bab content_3f5ead75f8469f56           1.0              3.0                  3.0
client_d211cb07b9059bab content_54a05859e320fb8f           7.0             17.0             2.428571
client_23a62021009f63c4 content_323d4c1e56a0ba87        

In [41]:
# Step 28: Recalculate impression-weighted average position

march_position = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_sum_position) AS sum_position_30d,
        SUM(gsc_impressions) AS impressions_for_position_30d
    FROM march
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

march_position["weighted_avg_position_30d"] = (
    march_position["sum_position_30d"]
    / march_position["impressions_for_position_30d"]
)

print("Rows with weighted position:", len(march_position))

print("\nWeighted position summary:")
print(
    march_position["weighted_avg_position_30d"].describe()
)

print("\nInvalid weighted positions:")
print(
    "Position <= 0:",
    (march_position["weighted_avg_position_30d"] <= 0).sum()
)

print(
    "Position > 100:",
    (march_position["weighted_avg_position_30d"] > 100).sum()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with weighted position: 176738

Weighted position summary:
count    176738.000000
mean         15.992270
std          18.097575
min           0.000000
25%           4.917879
50%           8.177966
75%          20.254025
max         309.000000
Name: weighted_avg_position_30d, dtype: float64

Invalid weighted positions:
Position <= 0: 1434
Position > 100: 103


In [42]:
# Step 29: Finalize decision-time feature frame

model_features = model_population.copy()

# Remove misleading / invalid ratio features
model_features = model_features.drop(
    columns=[
        "ai_traffic_share_30d",
        "scroll_rate_30d",
        "avg_position_30d",
    ],
    errors="ignore"
)

# Add impression-weighted average position
model_features = model_features.merge(
    march_position[
        [
            "client_hash_id",
            "content_hash_id",
            "weighted_avg_position_30d",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Treat zero position as missing rather than a real ranking position
model_features.loc[
    model_features["weighted_avg_position_30d"] <= 0,
    "weighted_avg_position_30d"
] = pd.NA

# CTR is a valid bounded ratio
model_features["ctr_30d"] = (
    model_features["clicks_30d"]
    / model_features["impressions_30d"].replace(0, pd.NA)
)

# Engagement rate is bounded when sessions exist
model_features["engagement_rate_30d"] = (
    model_features["engaged_sessions_30d"]
    / model_features["sessions_30d"].replace(0, pd.NA)
)

print("Final feature-frame rows:", len(model_features))

print("\nWeighted position missing:")
print(
    model_features["weighted_avg_position_30d"].isna().sum()
)

print("\nCTR > 100%:")
print(
    (model_features["ctr_30d"] > 1).sum()
)

print("\nEngagement rate > 100%:")
print(
    (model_features["engagement_rate_30d"] > 1).sum()
)

print("\nFeature frame shape:")
print(model_features.shape)

Final feature-frame rows: 251059

Weighted position missing:
133089

CTR > 100%:
0

Engagement rate > 100%:
0

Feature frame shape:
(251059, 34)


In [43]:
# Step 30: Build the future decline target from April

april_target = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april_30d,
        MAX(
            CASE
                WHEN gsc_data_available IS TRUE THEN 1
                ELSE 0
            END
        ) AS has_april_gsc
    FROM april
    GROUP BY client_hash_id, content_hash_id
""").df()

# Join future outcome to the March decision-time population
target_frame = model_features.merge(
    april_target,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Only pages with measurable GSC data in both periods can receive a target
target_frame["target_observable"] = (
    (target_frame["has_gsc_data"] == 1)
    & (target_frame["has_april_gsc"] == 1)
    & (target_frame["impressions_30d"] >= 500)
)

# Future decline: April impressions are at least 20% lower than March
target_frame["is_declining"] = (
    target_frame["target_observable"]
    & (
        target_frame["impressions_april_30d"]
        <= 0.80 * target_frame["impressions_30d"]
    )
)

print("Total decision-time pages:", len(target_frame))

print("\nTarget-observable pages:")
print(target_frame["target_observable"].sum())

print("\nNon-observable pages:")
print((~target_frame["target_observable"]).sum())

print("\nDeclining pages:")
print(target_frame["is_declining"].sum())

print("\nDecline rate among observable pages:")
print(
    target_frame.loc[
        target_frame["target_observable"],
        "is_declining"
    ].mean()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total decision-time pages: 251059

Target-observable pages:
41863

Non-observable pages:
209196

Declining pages:
22945

Decline rate among observable pages:
0.5480973652151064


In [44]:
# Step 31: Validate the future decline target

observable = target_frame[
    target_frame["target_observable"]
].copy()

# Actual percentage change from March to April
observable["impression_change_pct"] = (
    (
        observable["impressions_april_30d"]
        - observable["impressions_30d"]
    )
    / observable["impressions_30d"]
) * 100

print("=== Target population ===")
print("Observable pages:", len(observable))

print("\nTarget distribution:")
print(
    observable["is_declining"]
    .value_counts()
    .rename(index={False: "Not declining", True: "Declining"})
)

print("\nTarget rate:")
print(observable["is_declining"].mean())

print("\n=== March impressions ===")
print(observable["impressions_30d"].describe())

print("\n=== April impressions ===")
print(observable["impressions_april_30d"].describe())

print("\n=== Impression change (%) ===")
print(observable["impression_change_pct"].describe())

print("\n=== Rule validation ===")

# Every positive target should satisfy the decline rule
positive_check = observable.loc[
    observable["is_declining"],
    "impressions_april_30d"
] <= (
    0.80 * observable.loc[
        observable["is_declining"],
        "impressions_30d"
    ]
)

print("Positive labels satisfying <=80% rule:",
      positive_check.all())

# Every negative target should NOT satisfy the decline rule
negative_check = observable.loc[
    ~observable["is_declining"],
    "impressions_april_30d"
] <= (
    0.80 * observable.loc[
        ~observable["is_declining"],
        "impressions_30d"
    ]
)

print("Negative labels incorrectly satisfying rule:",
      negative_check.sum())

=== Target population ===
Observable pages: 41863

Target distribution:
is_declining
Declining        22945
Not declining    18918
Name: count, dtype: int64

Target rate:
0.5480973652151064

=== March impressions ===
count     41863.000000
mean       4604.037073
std        9153.967820
min         500.000000
25%         971.000000
50%        2027.000000
75%        4957.000000
max      617124.000000
Name: impressions_30d, dtype: float64

=== April impressions ===
count     41863.000000
mean       4153.307121
std        9513.374355
min           1.000000
25%         663.000000
50%        1536.000000
75%        4012.500000
max      799358.000000
Name: impressions_april_30d, dtype: float64

=== Impression change (%) ===
count    41863.000000
mean       -10.884926
std         74.872198
min        -99.980522
25%        -48.606811
50%        -24.704579
75%          5.170455
max       4262.038092
Name: impression_change_pct, dtype: float64

=== Rule validation ===
Positive labels satisfying <=8

In [45]:
# Step 32: Prepare the supervised modeling population

model_data = target_frame[
    target_frame["target_observable"]
].copy()

# Final target
model_data["target"] = model_data["is_declining"].astype(int)

print("Supervised modeling rows:", len(model_data))
print("Positive target:", model_data["target"].sum())
print("Negative target:", (model_data["target"] == 0).sum())

print("\nTarget rate:")
print(model_data["target"].mean())

print("\nAvailable feature columns:")
print(len(model_data.columns))

Supervised modeling rows: 41863
Positive target: 22945
Negative target: 18918

Target rate:
0.5480973652151064

Available feature columns:
39


In [46]:
# Step 33: Build a transparent rule-based baseline score

baseline = model_data.copy()

# 1. Visibility risk:
# Higher visibility means a page has more meaningful search exposure.
baseline["visibility_score"] = (
    baseline["impressions_30d"]
    .rank(pct=True)
)

# 2. Freshness risk:
# Older content gets a higher refresh priority.
baseline["freshness_risk_score"] = (
    baseline["content_age_days"]
    .rank(pct=True)
)

# 3. Position opportunity:
# Higher average position number = weaker ranking position.
# Missing position means no usable GSC position signal.
position_rank = baseline["weighted_avg_position_30d"].rank(
    pct=True,
    na_option="keep"
)

baseline["position_opportunity_score"] = position_rank.fillna(0)

# 4. CTR weakness:
# Lower CTR = higher potential review priority.
ctr_rank = baseline["ctr_30d"].rank(
    pct=True,
    ascending=True
)

baseline["ctr_weakness_score"] = ctr_rank

# Final transparent baseline
baseline["baseline_score"] = 100 * (
    0.35 * baseline["visibility_score"]
    + 0.25 * baseline["freshness_risk_score"]
    + 0.25 * baseline["position_opportunity_score"]
    + 0.15 * baseline["ctr_weakness_score"]
)

print("Baseline rows:", len(baseline))

print("\nBaseline score summary:")
print(
    baseline["baseline_score"].describe()
)

print("\nMissing baseline scores:")
print(
    baseline["baseline_score"].isna().sum()
)

print("\nTop 10 baseline candidates:")
print(
    baseline[
        [
            "content_hash_id",
            "impressions_30d",
            "content_age_days",
            "weighted_avg_position_30d",
            "ctr_30d",
            "baseline_score",
            "target",
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
    .to_string(index=False)
)

Baseline rows: 41863

Baseline score summary:
count    41863.000000
mean        50.001194
std         13.535552
min          3.223192
25%         40.753679
50%         50.548874
75%         60.120632
max         89.896388
Name: baseline_score, dtype: float64

Missing baseline scores:
0

Top 10 baseline candidates:
         content_hash_id  impressions_30d  content_age_days  weighted_avg_position_30d   ctr_30d  baseline_score  target
content_1302930cfeaa904e          35861.0               434                  17.618555   0.00382       89.896388       1
content_99d2d91a92668c8f          16882.0               434                  12.309797  0.005746       88.337852       1
content_3e7decc1c3c02394          10489.0               434                  16.561064  0.004672       87.727588       0
content_7960401fa9cfe53b          31135.0               263                  31.438285  0.007259       87.488653       0
content_4293c217789841d2          10675.0               417                  13

In [47]:
# Step 34: Evaluate the baseline using Precision@50

baseline_ranked = baseline.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

top_50 = baseline_ranked.head(50)

baseline_precision_at_50 = top_50["target"].mean()

print("Baseline Precision@50:", baseline_precision_at_50)
print("Declining pages in top 50:", top_50["target"].sum())
print("Top-50 pages evaluated:", len(top_50))

print("\nOverall positive rate:", baseline["target"].mean())

print(
    "\nBaseline lift over overall positive rate:",
    baseline_precision_at_50 / baseline["target"].mean()
)

Baseline Precision@50: 0.24
Declining pages in top 50: 12
Top-50 pages evaluated: 50

Overall positive rate: 0.5480973652151064

Baseline lift over overall positive rate: 0.4378784048812377


In [48]:
# Step 35: Final feature leakage audit

decision_date = pd.Timestamp("2026-03-31")

leakage_audit = pd.DataFrame({
    "feature": [
        "impressions_30d",
        "clicks_30d",
        "weighted_avg_position_30d",
        "pageviews_30d",
        "sessions_30d",
        "users_30d",
        "engaged_sessions_30d",
        "ai_sessions_30d",
        "scroll_events_30d",
        "ctr_30d",
        "engagement_rate_30d",
        "content_age_days",
        "content_created_date",
        "content_updated_date",
        "last_optimized_date",
        "optimization_eligible_date",
        "impressions_april_30d",
        "is_declining",
        "target",
    ],
    "type": [
        "March",
        "March",
        "March",
        "March",
        "March",
        "March",
        "March",
        "March",
        "March",
        "Derived from March",
        "Derived from March",
        "Derived from March",
        "Metadata",
        "Metadata",
        "Metadata",
        "Metadata",
        "April",
        "April",
        "Derived from April",
    ]
})

print(leakage_audit.to_string(index=False))

print("\nFuture-dated metadata:")
for col in [
    "content_created_date",
    "content_updated_date",
    "last_optimized_date",
    "optimization_eligible_date",
]:
    if col in model_data.columns:
        dates = pd.to_datetime(model_data[col], errors="coerce")
        future_count = (dates > decision_date).sum()
        print(f"{col}: {future_count}")

                   feature               type
           impressions_30d              March
                clicks_30d              March
 weighted_avg_position_30d              March
             pageviews_30d              March
              sessions_30d              March
                 users_30d              March
      engaged_sessions_30d              March
           ai_sessions_30d              March
         scroll_events_30d              March
                   ctr_30d Derived from March
       engagement_rate_30d Derived from March
          content_age_days Derived from March
      content_created_date           Metadata
      content_updated_date           Metadata
       last_optimized_date           Metadata
optimization_eligible_date           Metadata
     impressions_april_30d              April
              is_declining              April
                    target Derived from April

Future-dated metadata:
content_created_date: 0
content_updated_date: 33703
last

In [49]:
# Step 36: Define leakage-safe ML features

feature_columns = [
    # Search performance
    "impressions_30d",
    "clicks_30d",
    "weighted_avg_position_30d",
    "ctr_30d",

    # Analytics performance
    "pageviews_30d",
    "sessions_30d",
    "users_30d",
    "engaged_sessions_30d",
    "engagement_rate_30d",
    "scroll_events_30d",
    "ai_sessions_30d",

    # Observation / availability
    "days_observed",
    "has_gsc_data",
    "has_ga4_data",

    # Content characteristics
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "backlinks",
    "category_count",
    "char_count",
    "word_count",
]

X = model_data[feature_columns].copy()
y = model_data["target"].copy()

print("Number of ML features:", len(feature_columns))
print("\nFeatures:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:2}. {col}")

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Number of ML features: 22

Features:
 1. impressions_30d
 2. clicks_30d
 3. weighted_avg_position_30d
 4. ctr_30d
 5. pageviews_30d
 6. sessions_30d
 7. users_30d
 8. engaged_sessions_30d
 9. engagement_rate_30d
10. scroll_events_30d
11. ai_sessions_30d
12. days_observed
13. has_gsc_data
14. has_ga4_data
15. content_age_days
16. search_volume
17. competition
18. cpc
19. backlinks
20. category_count
21. char_count
22. word_count

X shape: (41863, 22)
y shape: (41863,)

Target distribution:
target
1    22945
0    18918
Name: count, dtype: int64


In [50]:
# Step 37: Final ML feature missingness check

missing_summary = (
    X.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / len(X) * 100
)

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

print("Missing-value summary:")
print(missing_summary.to_string())

print("\nTotal rows:", len(X))
print("Features with missing values:",
      (missing_summary["missing_count"] > 0).sum())

Missing-value summary:
                           missing_count  missing_pct
backlinks                          21890    52.289611
engagement_rate_30d                16530    39.485942
word_count                         11444    27.336789
char_count                         11444    27.336789
pageviews_30d                      11241    26.851874
sessions_30d                       11241    26.851874
ai_sessions_30d                    11241    26.851874
users_30d                          11241    26.851874
engaged_sessions_30d               11241    26.851874
scroll_events_30d                  11241    26.851874
competition                          565     1.349640
cpc                                  565     1.349640
search_volume                        565     1.349640
clicks_30d                             0     0.000000
weighted_avg_position_30d              0     0.000000
ctr_30d                                0     0.000000
impressions_30d                        0     0.000000
days_

In [52]:
# Step 38: Client-aware split + leakage-safe preprocessing

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Make a clean numeric feature matrix
X = model_data[feature_columns].copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y = model_data["target"].astype(int)
groups = model_data["client_hash_id"]

# Client-aware 80/20 split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTrain clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print(
    "\nClient overlap:",
    len(set(groups_train) & set(groups_test))
)

print("\nTrain target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

# Median imputation + missingness indicators
preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Train rows: 41418
Test rows: 445

Train clients: 22
Test clients: 6

Client overlap: 0

Train target rate: 0.5460910715148003
Test target rate: 0.7348314606741573

Processed train shape: (41418, 35)
Processed test shape: (445, 35)


In [53]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

splits = list(sgkf.split(X, y, groups=groups))

print("Number of folds:", len(splits))

for i, (train_idx, test_idx) in enumerate(splits, start=1):
    fold_train = y.iloc[train_idx]
    fold_test = y.iloc[test_idx]
    group_train = groups.iloc[train_idx]
    group_test = groups.iloc[test_idx]

    print(f"\nFold {i}")
    print("Train rows:", len(train_idx))
    print("Test rows:", len(test_idx))
    print("Train clients:", group_train.nunique())
    print("Test clients:", group_test.nunique())
    print("Client overlap:", len(set(group_train) & set(group_test)))
    print("Train target rate:", round(fold_train.mean(), 4))
    print("Test target rate:", round(fold_test.mean(), 4))

Number of folds: 5

Fold 1
Train rows: 41802
Test rows: 61
Train clients: 27
Test clients: 1
Client overlap: 0
Train target rate: 0.5483
Test target rate: 0.4262

Fold 2
Train rows: 35283
Test rows: 6580
Train clients: 22
Test clients: 6
Client overlap: 0
Train target rate: 0.5569
Test target rate: 0.5008

Fold 3
Train rows: 33165
Test rows: 8698
Train clients: 26
Test clients: 2
Client overlap: 0
Train target rate: 0.4988
Test target rate: 0.7361

Fold 4
Train rows: 16165
Test rows: 25698
Train clients: 13
Test clients: 15
Client overlap: 0
Train target rate: 0.6466
Test target rate: 0.4861

Fold 5
Train rows: 41037
Test rows: 826
Train clients: 24
Test clients: 4
Client overlap: 0
Train target rate: 0.5414
Test target rate: 0.8826


In [54]:
# Use Fold 2 as the final client-aware train/test split

train_idx, test_idx = splits[1]

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Rebuild preprocessing using training data only
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Final train rows:", len(X_train))
print("Final test rows:", len(X_test))

print("\nFinal train clients:", groups_train.nunique())
print("Final test clients:", groups_test.nunique())

print("\nClient overlap:", len(set(groups_train) & set(groups_test)))

print("\nFinal train target rate:", round(y_train.mean(), 4))
print("Final test target rate:", round(y_test.mean(), 4))

print("\nProcessed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Final train rows: 35283
Final test rows: 6580

Final train clients: 22
Final test clients: 6

Client overlap: 0

Final train target rate: 0.5569
Final test target rate: 0.5008

Processed train shape: (35283, 35)
Processed test shape: (6580, 35)


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Logistic Regression
logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# 2. Decision Tree
tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

# 3. Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

# Train all models
logistic_model.fit(X_train_processed, y_train)
tree_model.fit(X_train_processed, y_train)
rf_model.fit(X_train_processed, y_train)

print("Logistic Regression: trained")
print("Decision Tree: trained")
print("Random Forest: trained")

Logistic Regression: trained
Decision Tree: trained
Random Forest: trained


In [56]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model
}

evaluation_results = []

for name, model in models.items():

    # Predictions
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    average_precision = average_precision_score(y_test, y_prob)

    # Precision@50
    top_50_idx = y_prob.argsort()[::-1][:50]
    precision_at_50 = y_test.iloc[top_50_idx].mean()

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "Average_Precision": average_precision,
        "Precision@50": precision_at_50
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df = evaluation_df.sort_values(
    by=["Precision@50", "Average_Precision", "ROC_AUC"],
    ascending=False
).reset_index(drop=True)

display(evaluation_df.round(4))

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision,Precision@50
0,Random Forest,0.3260,0.3324,0.3429,0.3376,0.2843,0.3861,0.54
1,Logistic Regression,0.3064,0.2874,0.2604,0.2732,0.2969,0.3866,0.44
2,Decision Tree,0.3602,0.3730,0.4079,0.3897,0.3949,0.4734,0.30


In [57]:
# Fair baseline comparison on the same held-out test set

test_baseline = model_data.iloc[test_idx].copy()

# Recreate the heuristic baseline score on the test set
test_baseline["visibility_score"] = (
    test_baseline["impressions_30d"].rank(pct=True)
)

test_baseline["freshness_risk_score"] = (
    test_baseline["content_age_days"].rank(pct=True)
)

position_rank = test_baseline["weighted_avg_position_30d"].rank(
    pct=True,
    na_option="keep"
)

test_baseline["position_opportunity_score"] = (
    position_rank.fillna(0)
)

test_baseline["ctr_weakness_score"] = (
    test_baseline["ctr_30d"].rank(
        pct=True,
        ascending=True
    )
)

test_baseline["baseline_score"] = 100 * (
    0.35 * test_baseline["visibility_score"]
    + 0.25 * test_baseline["freshness_risk_score"]
    + 0.25 * test_baseline["position_opportunity_score"]
    + 0.15 * test_baseline["ctr_weakness_score"]
)

# Baseline Precision@50
baseline_top_50 = (
    test_baseline
    .sort_values("baseline_score", ascending=False)
    .head(50)
)

baseline_precision_at_50 = baseline_top_50["target"].mean()

# Random Forest Precision@50
rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

rf_top_50_idx = rf_prob.argsort()[::-1][:50]
rf_precision_at_50 = y_test.iloc[rf_top_50_idx].mean()

comparison_df = pd.DataFrame({
    "Approach": [
        "Heuristic Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision_at_50,
        rf_precision_at_50
    ]
})

comparison_df["Lift_vs_overall_rate"] = (
    comparison_df["Precision@50"] / y_test.mean()
)

display(comparison_df.round(4))

print("\nOverall test positive rate:", round(y_test.mean(), 4))
print("Baseline Precision@50:", round(baseline_precision_at_50, 4))
print("Random Forest Precision@50:", round(rf_precision_at_50, 4))

,Approach,Precision@50,Lift_vs_overall_rate
0,Heuristic Baseline,0.40,0.7988
1,Random Forest,0.54,1.0784



Overall test positive rate: 0.5008
Baseline Precision@50: 0.4
Random Forest Precision@50: 0.54


In [58]:
# ---------------------------------------------------------
# Final decision-time scoring population
# ---------------------------------------------------------
# Important:
# Do NOT use April information here.
# Only March decision-time information is allowed.

scoring_population = model_features[
    (model_features["has_gsc_data"] == 1) &
    (model_features["impressions_30d"] >= 500)
].copy()

print("Final scoring population:", len(scoring_population))

# Prepare model features
X_score = scoring_population[feature_columns].copy()

for col in X_score.columns:
    X_score[col] = pd.to_numeric(X_score[col], errors="coerce")

# Apply the already-fitted preprocessing pipeline
X_score_processed = preprocessor.transform(X_score)

# Predict probability of future decline
scoring_population["decline_probability"] = (
    rf_model.predict_proba(X_score_processed)[:, 1]
)

# Rank highest-risk pages first
scoring_population = scoring_population.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

scoring_population["priority_rank"] = (
    scoring_population.index + 1
)

print("Ranked pages:", len(scoring_population))

display(
    scoring_population[
        [
            "priority_rank",
            "client_hash_id",
            "content_hash_id",
            "decline_probability",
            "impressions_30d",
            "clicks_30d",
            "ctr_30d",
            "weighted_avg_position_30d",
            "content_age_days"
        ]
    ].head(20).round(4)
)

Final scoring population: 41912
Ranked pages: 41912


,priority_rank,client_hash_id,content_hash_id,decline_probability,impressions_30d,clicks_30d,ctr_30d,weighted_avg_position_30d,content_age_days
0,1,client_62f4a7e64f5e0096,content_1712783cce40c318,0.9288,704.0,0.0,0.0,15.0369,266
1,2,client_62f4a7e64f5e0096,content_2dfe1f0e77f425b6,0.9288,901.0,0.0,0.0,11.9023,266
2,3,client_62f4a7e64f5e0096,content_f41be5dd1e97e696,0.9276,749.0,0.0,0.0,17.7557,278
3,4,client_62f4a7e64f5e0096,content_fc3e06561e553793,0.9207,937.0,0.0,0.0,7.1003,266
4,5,client_62f4a7e64f5e0096,content_ac2d39755783c25a,0.9206,786.0,0.0,0.0,21.5471,266
5,6,client_62f4a7e64f5e0096,content_d15821219d90b295,0.9205,956.0,0.0,0.0,10.2751,266
6,7,client_62f4a7e64f5e0096,content_e67c8fe65568a0c2,0.9193,718.0,0.0,0.0,5.0515,266
7,8,client_62f4a7e64f5e0096,content_c51ea47779491fd5,0.9179,870.0,0.0,0.0,34.8805,266
8,9,client_62f4a7e64f5e0096,content_63993129c83babc1,0.9168,802.0,0.0,0.0,11.3903,266
9,10,client_62f4a7e64f5e0096,content_3fcd66725e948060,0.9163,736.0,0.0,0.0,14.9402,266


In [59]:
# ---------------------------------------------------------
# Reason codes for ranked recommendations
# ---------------------------------------------------------

def generate_reason_codes(row):
    reasons = []

    # CTR weakness
    if pd.notna(row["ctr_30d"]) and row["ctr_30d"] <= 0.01:
        reasons.append("Very low CTR")

    # Search visibility
    if row["impressions_30d"] >= 1000:
        reasons.append("High search visibility")

    # Search position opportunity
    if (
        pd.notna(row["weighted_avg_position_30d"])
        and row["weighted_avg_position_30d"] >= 10
    ):
        reasons.append("Weak average search position")

    # Content age
    if row["content_age_days"] >= 180:
        reasons.append("Mature content")

    # Engagement
    if (
        pd.notna(row["engagement_rate_30d"])
        and row["engagement_rate_30d"] < 0.5
    ):
        reasons.append("Low engagement rate")

    # Content depth
    if (
        pd.notna(row["word_count"])
        and row["word_count"] >= 2000
    ):
        reasons.append("Long-form content")

    if not reasons:
        reasons.append("Multiple historical performance signals")

    return "; ".join(reasons[:4])


scoring_population["reason_codes"] = (
    scoring_population.apply(generate_reason_codes, axis=1)
)

# Recommended action
def recommend_action(row):
    reasons = row["reason_codes"]

    if "Very low CTR" in reasons:
        return "Review title/meta and search-intent alignment"

    if "Weak average search position" in reasons:
        return "Review topical relevance and on-page optimization"

    if "Low engagement rate" in reasons:
        return "Review content relevance and user experience"

    return "Prioritize for content refresh review"


scoring_population["recommended_action"] = (
    scoring_population.apply(recommend_action, axis=1)
)

# Show top recommendations
top_recommendations = scoring_population[
    [
        "priority_rank",
        "client_hash_id",
        "content_hash_id",
        "decline_probability",
        "impressions_30d",
        "ctr_30d",
        "weighted_avg_position_30d",
        "content_age_days",
        "reason_codes",
        "recommended_action"
    ]
].head(20)

display(top_recommendations.round(4))

,priority_rank,client_hash_id,content_hash_id,decline_probability,impressions_30d,ctr_30d,weighted_avg_position_30d,content_age_days,reason_codes,recommended_action
0,1,client_62f4a7e64f5e0096,content_1712783cce40c318,0.9288,704.0,0.0,15.0369,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
1,2,client_62f4a7e64f5e0096,content_2dfe1f0e77f425b6,0.9288,901.0,0.0,11.9023,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
2,3,client_62f4a7e64f5e0096,content_f41be5dd1e97e696,0.9276,749.0,0.0,17.7557,278,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
3,4,client_62f4a7e64f5e0096,content_fc3e06561e553793,0.9207,937.0,0.0,7.1003,266,Very low CTR; Mature content,Review title/meta and search-intent alignment
4,5,client_62f4a7e64f5e0096,content_ac2d39755783c25a,0.9206,786.0,0.0,21.5471,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
5,6,client_62f4a7e64f5e0096,content_d15821219d90b295,0.9205,956.0,0.0,10.2751,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
6,7,client_62f4a7e64f5e0096,content_e67c8fe65568a0c2,0.9193,718.0,0.0,5.0515,266,Very low CTR; Mature content,Review title/meta and search-intent alignment
7,8,client_62f4a7e64f5e0096,content_c51ea47779491fd5,0.9179,870.0,0.0,34.8805,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
8,9,client_62f4a7e64f5e0096,content_63993129c83babc1,0.9168,802.0,0.0,11.3903,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
9,10,client_62f4a7e64f5e0096,content_3fcd66725e948060,0.9163,736.0,0.0,14.9402,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment


In [60]:
# ---------------------------------------------------------
# Export final ranked recommendations
# ---------------------------------------------------------

final_ranking = scoring_population[
    [
        "priority_rank",
        "client_hash_id",
        "content_hash_id",
        "decline_probability",
        "impressions_30d",
        "clicks_30d",
        "ctr_30d",
        "weighted_avg_position_30d",
        "content_age_days",
        "reason_codes",
        "recommended_action"
    ]
].copy()

output_path = "final_content_refresh_ranking.csv"

final_ranking.to_csv(
    output_path,
    index=False
)

print("Exported:", output_path)
print("Rows:", len(final_ranking))
print("Columns:", len(final_ranking.columns))

print("\nTop 10 recommendations:")
display(final_ranking.head(10).round(4))

Exported: final_content_refresh_ranking.csv
Rows: 41912
Columns: 11

Top 10 recommendations:


,priority_rank,client_hash_id,content_hash_id,decline_probability,impressions_30d,clicks_30d,ctr_30d,weighted_avg_position_30d,content_age_days,reason_codes,recommended_action
0,1,client_62f4a7e64f5e0096,content_1712783cce40c318,0.9288,704.0,0.0,0.0,15.0369,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
1,2,client_62f4a7e64f5e0096,content_2dfe1f0e77f425b6,0.9288,901.0,0.0,0.0,11.9023,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
2,3,client_62f4a7e64f5e0096,content_f41be5dd1e97e696,0.9276,749.0,0.0,0.0,17.7557,278,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
3,4,client_62f4a7e64f5e0096,content_fc3e06561e553793,0.9207,937.0,0.0,0.0,7.1003,266,Very low CTR; Mature content,Review title/meta and search-intent alignment
4,5,client_62f4a7e64f5e0096,content_ac2d39755783c25a,0.9206,786.0,0.0,0.0,21.5471,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
5,6,client_62f4a7e64f5e0096,content_d15821219d90b295,0.9205,956.0,0.0,0.0,10.2751,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
6,7,client_62f4a7e64f5e0096,content_e67c8fe65568a0c2,0.9193,718.0,0.0,0.0,5.0515,266,Very low CTR; Mature content,Review title/meta and search-intent alignment
7,8,client_62f4a7e64f5e0096,content_c51ea47779491fd5,0.9179,870.0,0.0,0.0,34.8805,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
8,9,client_62f4a7e64f5e0096,content_63993129c83babc1,0.9168,802.0,0.0,0.0,11.3903,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment
9,10,client_62f4a7e64f5e0096,content_3fcd66725e948060,0.9163,736.0,0.0,0.0,14.9402,266,Very low CTR; Weak average search position; Ma...,Review title/meta and search-intent alignment


In [61]:
# ---------------------------------------------------------
# Final artifact quality checks
# ---------------------------------------------------------

print("Total rows:", len(final_ranking))

print(
    "Rank sequence valid:",
    final_ranking["priority_rank"].equals(
        pd.Series(range(1, len(final_ranking) + 1))
    )
)

print(
    "Duplicate client-page pairs:",
    final_ranking.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

print(
    "Missing decline probabilities:",
    final_ranking["decline_probability"].isna().sum()
)

print(
    "Invalid probabilities:",
    (
        (final_ranking["decline_probability"] < 0) |
        (final_ranking["decline_probability"] > 1)
    ).sum()
)

print(
    "Missing reason codes:",
    final_ranking["reason_codes"].isna().sum()
)

print(
    "Missing recommended actions:",
    final_ranking["recommended_action"].isna().sum()
)

print(
    "Probability sorted descending:",
    final_ranking["decline_probability"].is_monotonic_decreasing
)

print("\nTop probability:", round(
    final_ranking["decline_probability"].iloc[0], 4
))

print("Bottom probability:", round(
    final_ranking["decline_probability"].iloc[-1], 4
))

Total rows: 41912
Rank sequence valid: True
Duplicate client-page pairs: 0
Missing decline probabilities: 0
Invalid probabilities: 0
Missing reason codes: 0
Missing recommended actions: 0
Probability sorted descending: True

Top probability: 0.9288
Bottom probability: 0.0842


## 2. Data

### Dataset and source

This capstone uses the pseudonymized FlyRank internship warehouse release
`flyrank_pseudonymized_warehouse_release_v20260703`, exported on 2026-07-03.

The analysis uses the following warehouse tables:

- `dim_clients` for client-level metadata.
- `dim_content` for content-page metadata.
- `fact_content_daily_performance` for daily search and traffic performance signals.

The warehouse contains daily content-performance data from 2025-01-27
through 2026-06-30. The final capstone analysis uses March 2026 as the
decision window and April 2026 as the future evaluation window.

### Decision-time population

The decision date is 2026-03-31. Only content that existed by this date
was retained. Pages younger than 90 days were excluded so that very new
content would not be treated as established content requiring refresh
prioritization.

For supervised target construction, pages needed at least 500 March
Google Search Console impressions and observable Google Search Console
coverage in both the decision and future periods.

The final deployment-style ranking does not use April information. It
scores March-visible mature pages with at least 500 March impressions.

### Public-safe handling

Client and content identifiers remain pseudonymized. No client names,
domains, private search queries, credentials, or raw warehouse exports are
included in the analysis output.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Decision design

The analysis uses a forward-looking refresh-prioritization setup.

- **Decision window:** March 1–31, 2026
- **Future outcome window:** April 1–30, 2026
- **Decision date:** March 31, 2026

The model is intended to answer a prioritization question: which existing
content pages should a content team review first based on signals available
at the decision date.

### Target definition

A page is considered to have a future visibility decline when:

1. It has at least 500 Google Search Console impressions during March.
2. Google Search Console coverage is observable in both the decision and
   future periods.
3. April impressions are at most 80% of March impressions.

The 500-impression threshold was selected after sensitivity analysis across
multiple minimum-impression thresholds. It provides a practical balance
between retaining enough pages and avoiding very low-volume observations.

### Feature engineering

Features were constructed only from information available during the March
decision window and from content metadata that was valid by the decision
date.

The final feature set contains 22 variables covering:

- Search visibility and clicks
- Click-through rate
- Impression-weighted average search position
- Pageviews, sessions, users, and engaged sessions
- Engagement rate
- Scroll events and AI sessions
- Days observed and data-availability indicators
- Content age
- Search volume, competition, and CPC
- Backlinks
- Content category count
- Character count and word count

Daily search position was replaced with an impression-weighted position
measure because a simple daily average can give equal weight to days with
very different impression volumes.

### Missing values and preprocessing

Missing numeric values were handled using median imputation. Missingness
indicators were also added so that the models could distinguish observed
values from imputed values.

The imputation and scaling pipeline was fitted on the training data only
and then applied to the held-out test data.

### Leakage prevention

Future-looking metadata was excluded from the final feature set. In
particular, fields containing dates after the March decision date were not
used as model features.

April performance data was used only to construct the future target and
evaluate the models. It was never included as an input feature.

### Validation design

A client-aware holdout was used so that clients represented in the test set
did not appear in the training set.

The final split contains:

- **Training:** 35,283 pages across 22 clients
- **Test:** 6,580 pages across 6 clients
- **Client overlap:** 0
- **Training decline rate:** 55.69%
- **Test decline rate:** 50.08%

This design provides a stricter test of whether the learned ranking can
generalize across clients rather than simply memorizing client-specific
patterns.

### Models

Three supervised classification models were evaluated:

1. Logistic Regression
2. Decision Tree
3. Random Forest

The Random Forest used 200 trees with a maximum depth of 10 and a minimum
leaf size of 25.

### Evaluation

Because the operational goal is to identify a small set of pages for human
review, **Precision@50** was selected as the primary business metric.

Additional evaluation metrics included:

- Accuracy
- Precision
- Recall
- F1 score
- ROC-AUC
- Average Precision

A heuristic baseline was evaluated on the same held-out test set to
determine whether the machine-learning ranking provided additional
prioritization value.

### Final ranking

After model comparison, the selected model was applied to the decision-time
scoring population: mature pages with March Google Search Console
visibility and at least 500 March impressions.

The final output ranks 41,912 pages by predicted probability of future
visibility decline. Each recommendation also includes rule-based reason
codes and a suggested review action.

These reason codes are intended as prioritization aids rather than causal
explanations.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Model performance

The models were evaluated on the same held-out client-aware test set of
6,580 pages.

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | Average Precision | Precision@50 |
|---|---:|---:|---:|---:|---:|---:|---:|
| Random Forest | 0.3260 | 0.3324 | 0.3429 | 0.3376 | 0.2843 | 0.3861 | **0.54** |
| Logistic Regression | 0.3064 | 0.2874 | 0.2604 | 0.2732 | 0.2969 | 0.3866 | 0.44 |
| Decision Tree | 0.3602 | 0.3730 | 0.4079 | 0.3897 | 0.3949 | 0.4734 | 0.30 |

The Random Forest was selected as the final ranking model because
Precision@50 is the primary metric for this prioritization task.

### Baseline comparison

The heuristic baseline achieved a Precision@50 of **0.40** on the same
held-out test set, while the Random Forest achieved **0.54**.

The overall decline rate in the test set was **0.5008**. Therefore:

- Heuristic baseline lift vs overall rate: **0.7988×**
- Random Forest lift vs overall rate: **1.0784×**
- Random Forest improvement over the baseline: **+14 percentage points**

The Random Forest therefore produced a modest improvement in top-50
prioritization over the heuristic baseline.

### Interpretation

The results provide evidence that historical performance signals can add
some value for prioritizing pages for refresh review. However, the overall
predictive performance is limited.

In particular, the Random Forest ROC-AUC was **0.2843**, below 0.50. The
model should therefore not be described as a strong general-purpose
classifier or as accurately predicting future decline for every page.

The more defensible interpretation is narrower: the Random Forest produced
a better top-50 prioritization result than the heuristic baseline on this
specific client-aware holdout.

Precision@50 is therefore emphasized over accuracy because the operational
question is which small set of pages should be reviewed first, rather than
whether every page can be classified correctly.

## 5. Limitations

This analysis should be interpreted as a prioritization experiment rather
than a definitive predictive system.

### Limited predictive strength

The Random Forest achieved a Precision@50 of 0.54, which was better than the
0.40 heuristic baseline on the held-out test set. However, its ROC-AUC was
0.2843, indicating weak overall ranking discrimination.

Therefore, the model should not be presented as a highly accurate predictor
of future content decline.

### Single future evaluation window

The target was defined using one forward month: April 2026. A single future
window cannot establish whether the observed ranking performance would remain
stable across different months, seasons, or search environments.

A stronger future study would evaluate multiple rolling decision windows.

### Client distribution

The validation design used a client-aware holdout so that test clients were
not present in training. This is stricter than a random page-level split,
but the available client histories and target distributions are uneven.

The selected holdout should therefore be treated as one practical
generalization test rather than proof that the model will perform equally
well for every client.

### Observability constraints

The supervised target could only be constructed for pages with observable
Google Search Console data in both the decision and future periods.
Consequently, the labeled modeling population is smaller than the full
decision-time population.

The final scoring population also includes a small number of pages for which
the future target could not yet be observed. These pages are ranked using
March-only information and are not treated as labeled evaluation examples.

### Feature limitations

The available warehouse fields provide historical performance and content
metadata, but they do not capture every factor that may affect future search
visibility.

Examples include search-intent changes, competitor actions, algorithm
updates, content quality changes, and editorial decisions.

### No causal interpretation

The model identifies statistical prioritization signals. It does not show
that any individual factor causes a page to decline.

The rule-based reason codes are therefore explanations for why a page was
flagged for review, not causal explanations of future performance.

### Operational interpretation

The output should be used as a review-prioritization aid for human editors.
It should not automatically trigger content changes, deletion, or other
irreversible actions.

The main evidence from this experiment is limited to whether historical
signals can improve the ordering of pages for a small review queue.

## 6. Ranked recommendations

The final Random Forest model was applied to the decision-time scoring
population using March 2026 information only.

This produced a ranked list of **41,912 content pages** by predicted
probability of future visibility decline.

### How to use the ranking

The ranking is intended to support a small editorial review queue.

Pages near the top of the ranking should be reviewed first, but the score
should be treated as a prioritization signal rather than a guarantee that
the page will decline.

Each ranked page includes:

- Priority rank
- Anonymized client identifier
- Anonymized content identifier
- Predicted decline probability
- March impressions and clicks
- March CTR
- Impression-weighted average search position
- Content age
- Rule-based reason codes
- Recommended review action

### Review playbook

The reason codes provide simple, human-readable signals that can help an
editor understand why a page was prioritized.

| Signal | Suggested review action |
|---|---|
| Very low CTR | Review title, metadata, and search-intent alignment |
| Weak average search position | Review topical relevance and on-page optimization |
| Low engagement rate | Review content relevance and user experience |
| Mature content | Check whether the page still reflects current user needs |
| High search visibility | Prioritize pages where a refresh could protect meaningful existing visibility |
| Multiple historical performance signals | Conduct a broader content refresh review |

These recommendations are deliberately framed as review actions rather than
automatic interventions.

### Top-ranked pages

The complete ranked output is stored in the project artifact
`final_content_refresh_ranking.csv`.

The artifact contains **41,912 ranked pages**, sorted from highest to lowest
predicted decline probability.

The top-ranked pages should be treated as the first candidates for human
review. Because the dataset is anonymized, the public research output uses
hash-based client and content identifiers rather than real client names,
domains, URLs, or private queries.

### Important interpretation note

A high predicted probability does not mean that a page will definitely
decline. It means that, according to the trained model and the historical
signals available at the March 31 decision date, the page received a higher
priority for review.

The ranking should therefore be used to allocate limited editorial review
capacity, not to replace editorial judgment.

## 7. Artifacts the paper embeds

### Final ranking artifact

The final deployment-style output was exported as:

`final_content_refresh_ranking.csv`

The artifact contains **41,912 ranked content pages** and **11 columns**.

The ranking is sorted by `decline_probability` in descending order and
contains:

- `priority_rank`
- `client_hash_id`
- `content_hash_id`
- `decline_probability`
- `impressions_30d`
- `clicks_30d`
- `ctr_30d`
- `weighted_avg_position_30d`
- `content_age_days`
- `reason_codes`
- `recommended_action`

### Final artifact validation

The exported artifact passed the following quality checks:

- Total rows: **41,912**
- Rank sequence valid: **True**
- Duplicate client-page pairs: **0**
- Missing decline probabilities: **0**
- Invalid probabilities: **0**
- Missing reason codes: **0**
- Missing recommended actions: **0**
- Probability sorted descending: **True**
- Top predicted decline probability: **0.9288**
- Bottom predicted decline probability: **0.0842**

### Recommended visualizations

The following outputs are useful for communicating the analysis:

1. Model comparison by Precision@50
2. Baseline versus Random Forest Precision@50
3. Distribution of predicted decline probabilities
4. Distribution of March-to-April impression change among observable pages
5. Top-ranked recommendation examples

All visualizations should use aggregate or anonymized information and
should not expose client names, domains, URLs, private queries, or other
sensitive identifiers.

### Reproducibility

The notebook contains the complete analysis workflow from warehouse access
and data validation through feature construction, target creation, model
training, evaluation, and final ranking export.

The final CSV is the primary machine-readable artifact produced by the
analysis.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
